# Main file 

05.11.2025

Written by: Arnav Kumar Upadhyay, Cedric Meyer adapted from MATLAB code of Marc, Jan Spindler DOI-https://doi.org/10.1021/acssynbio.4c00578

This script is the main script for the TXTL v2 model in INDEPENDENT GENE SIMULATION mode.

The script simulates each gene independently without competition for shared resources.


## Settings

In [31]:
# Packages functions and data types --------
include("packages.jl") 

# Sequences 
sequence_file	= "lacS.fa"
hybrid_file		= "lacS.txt"

# initialization of data containers
scale = txtl_types.init_scale()         # struct that makes it possible to influence rates by scaling them
tags = txtl_types.init_data()           # data container saving everything necessary to start the ODE solver 
tags.growth_rate = 1                    # A growth rate (0.1 to 1.8) can be chosen and the proteins of the TXTL machinery will be tuned according to that growth rate and the processivity of the RNA polymerase

## flags for efficiency factors
positive_aminoacid	= true 
anti_shinedalgarno	= true      # If no hybridization file is available put flag to false. We apply the intaRNA program’s intarna function (q: aSD, t: sequence, SeedBP: 5, out: tMinE) to generate a hybridization profile along the sequence. Link to the website: https://rna.informatik.uni-freiburg.de/IntaRNA/Input.jsp;jsessionid=2AE257E91E7276283F22CD77A5A8AA03
dynamic_efficiency	= true 

## Import information about model species and sequences 
seqs, seq_names, hyb_file_dict = species_and_sequences!(tags, sequence_file, hybrid_file)

## Calculate efficiency factors 
efac_pos_aa_dict, efac_int_sd_dict = efficiency_factor_initialization!(tags, seqs, seq_names)

## calculation of initial concentrations
y0 = calculation_of_initial_concentration!(tags, seqs, seq_names);

## calculate and load kinetic parameters
parameters!(tags);

## Release factor balance
RF_balance!(y0, tags)

# Preintegration
## Definition of mass matrix
conserved = [tags.idx["EFG"], tags.idx["EFD"]] # EFG, EFTu_GDP - Conserved species, Algebraic equations are given by mass balances
M = Matrix{Float64}(I, length(y0), length(y0)) # Creates unity matrix in the size of y0
M[conserved, :] .= 0; # setting mass matrix to zero to  remove time derivatives from those variables. This converts these differential equations into algebraic constraints

## Integration of the DAE system without DNA in the system 
y_steady_state = preintegration(tags, y0, M)

# Elongation factor balance 
EF_balance!(tags, y0, y_steady_state)

# Build states vector for multiple sequences 
y0_all = combined_states_vector(tags, y_steady_state, seqs, seq_names)

# Prepare per-sequence tags (for use in ODE)
local_tags_list = generate_local_tags(seqs, seq_names, tags, efac_pos_aa_dict, efac_int_sd_dict);

# Integration
conserved = [tags.idx["EFG"], tags.idx["EFD"]]  # EFG, EFTu_GDP - Conserved species
M = Matrix{Float64}(I, length(y0_all), length(y0_all))
M[conserved, :] .= 0

tspan 	= (0.0, 100.0) ## Define time span for the simulation
p_parallel = (local_tags_list,)
f 		= ODEFunction(ODE_parallel_distributed!, mass_matrix = M) ## Define ODEFunction to make use of mass matrix
prob 	= ODEProblem(f, y0_all, tspan, p_parallel) ## Create an ODE problem

# --- Timing the integration step ---
println("\n⏳ Integrating ODE system (distributed, test set)...")

sol = solve(prob, FBDF(autodiff=false), reltol = 1e-10, abstol = 1e-6) ## Solve ODE system

✅ Successfully loaded 1 sequences

⏳ Integrating ODE system (distributed, test set)...


retcode: Success
Interpolation: 3rd order Hermite
t: 231-element Vector{Float64}:
   0.0
   1.639285457830344e-9
   7.858580010003785e-9
   1.9974915655968084e-8
   3.68930571113165e-8
   1.5049488864079002e-7
   3.8727420930144846e-7
   6.765961072378633e-7
   9.659180051742782e-7
   1.7230186834039004e-6
   2.68374332444817e-6
   3.6444679654924397e-6
   4.714537012798672e-6
   ⋮
   1.0067259109351994
   1.127270466155716
   1.2738742338994462
   1.4746543578855356
   1.7439635310251356
   2.1302057728563986
   2.801128910752221
   4.629430708787655
   8.815531885518414
  19.355345622035784
  53.22207754642958
 100.0
u: 231-element Vector{Vector{Float64}}:
 [20.066985600703717, 20.086056744926683, 0.011764823802035312, 0.020942576143074654, 2.3889195399692102, 1.5446822292047273, 0.6637432999699595, 0.7463944193090334, 0.21061805577608217, 2.3619242296179084  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [20.066985600703717, 20.086056744926683, 0.011764823802035312, 0.0209425